# Experiment 7: Dimensionality Reduction and Model Evaluation (With and Without PCA)

Wisconsin Diagnostic Breast Cancer dataset (569 samples, 30 features). Ten classifiers evaluated with and without PCA (95% variance target -> 10 components).

**Note:** `xgboost` is not installable in this environment (no network access). `HistGradientBoostingClassifier` (scikit-learn's built-in histogram-based gradient boosting) is used as a comparable substitute, labeled `XGBoost (HistGB)`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
                               HistGradientBoostingClassifier, StackingClassifier)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix)

sns.set_style("whitegrid")
rng = 42

## 1. Load Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")
target_names = data.target_names

print("Shape:", X.shape)
print("Missing values:", X.isnull().sum().sum())
y.map({0: "malignant", 1: "benign"}).value_counts()

## 2. Preprocess: Split and Standardize

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rng)

scaler = StandardScaler()
X_train_std = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_std = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("Train:", X_train_std.shape, " Test:", X_test_std.shape)

## 3. PCA -- 95% Explained Variance Target

In [ ]:
pca_full = PCA(random_state=rng)
pca_full.fit(X_train_std)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.argmax(cum_var >= 0.95) + 1)
print(f"{n_components_95} components explain {cum_var[n_components_95-1]*100:.2f}% variance")

plt.figure(figsize=(7,4.5))
plt.plot(range(1, len(cum_var)+1), cum_var*100, "o-")
plt.axhline(95, color="red", linestyle="--", label="95% variance target")
plt.axvline(n_components_95, color="green", linestyle="--", label=f"{n_components_95} components")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance (%)")
plt.title("PCA Scree Plot -- Cumulative Explained Variance")
plt.legend()
plt.show()

In [ ]:
pca = PCA(n_components=n_components_95, random_state=rng)
X_train_pca = pd.DataFrame(pca.fit_transform(X_train_std), index=X_train_std.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_std), index=X_test_std.index)
print("PCA-reduced shape:", X_train_pca.shape)

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=y.map({0: "malignant", 1: "benign"}))
plt.title("Class Distribution -- Breast Cancer Wisconsin (Diagnostic)")
plt.show()

## 5. Helper Functions

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rng)

def test_metrics(model, Xtr, ytr, Xte, yte):
    t0 = time.time()
    model.fit(Xtr, ytr)
    train_time = time.time() - t0
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1] if hasattr(model, "predict_proba") else None
    return dict(
        Accuracy=accuracy_score(yte, pred),
        Precision=precision_score(yte, pred),
        Recall=recall_score(yte, pred),
        F1=f1_score(yte, pred),
        ROC_AUC=roc_auc_score(yte, proba) if proba is not None else None,
        TrainTime=train_time,
    ), pred, proba

def run_model(name, estimator, param_grid, X_train_set, y_train_set, X_test_set, y_test_set):
    if param_grid:
        grid = GridSearchCV(estimator, param_grid=param_grid, cv=skf, scoring="accuracy", n_jobs=-1)
        grid.fit(X_train_set, y_train_set)
        best_est, best_params, best_cv_acc = grid.best_estimator_, grid.best_params_, grid.best_score_
    else:
        best_est = estimator
        best_est.fit(X_train_set, y_train_set)
        best_params = {}
        best_cv_acc = cross_val_score(best_est, X_train_set, y_train_set, cv=skf, scoring="accuracy").mean()
    fold_scores = cross_val_score(best_est, X_train_set, y_train_set, cv=skf, scoring="accuracy")
    metrics, pred, proba = test_metrics(best_est, X_train_set, y_train_set, X_test_set, y_test_set)
    print(f"  {name}: best_params={best_params}, CV_acc={best_cv_acc:.4f}, test_acc={metrics['Accuracy']:.4f}")
    return {"best_params": best_params, "cv_fold_scores": fold_scores, "cv_avg": fold_scores.mean(),
            "cv_std": fold_scores.std(), "test_metrics": metrics}, best_est, pred, proba

## 6. Define All 9 Grid-Searched Models

In [ ]:
model_defs = {
    "SVM": (SVC(probability=True, random_state=rng),
            [{"kernel": ["linear"], "C": [0.1, 1, 10]},
             {"kernel": ["rbf"], "C": [0.1, 1, 10], "gamma": ["scale", "auto"]}]),
    "Naive Bayes": (GaussianNB(), {"var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]}),
    "KNN": (KNeighborsClassifier(), {"n_neighbors": [3, 5, 7, 9, 11], "weights": ["uniform", "distance"],
                                       "metric": ["euclidean", "manhattan"]}),
    "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=rng),
                             {"C": [0.01, 0.1, 1, 10, 100]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=rng),
                       {"max_depth": [3, 5, 7, None], "min_samples_leaf": [1, 2, 4]}),
    "Random Forest": (RandomForestClassifier(random_state=rng),
                       {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None]}),
    "AdaBoost": (AdaBoostClassifier(random_state=rng),
                 {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 0.5, 1.0]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=rng),
                           {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
    "XGBoost (HistGB)": (HistGradientBoostingClassifier(random_state=rng),
                          {"max_iter": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]}),
}

## 7. Train Every Model -- No-PCA and With-PCA

In [ ]:
results_no_pca, results_with_pca = {}, {}
fitted_no_pca, fitted_with_pca = {}, {}
preds_no_pca, preds_with_pca = {}, {}
probas_no_pca, probas_with_pca = {}, {}

for name, (estimator, grid) in model_defs.items():
    print(f"[No-PCA] {name}")
    res, best_est, pred, proba = run_model(name, estimator, grid, X_train_std, y_train, X_test_std, y_test)
    results_no_pca[name] = res
    fitted_no_pca[name], preds_no_pca[name], probas_no_pca[name] = best_est, pred, proba

    print(f"[With-PCA] {name}")
    estimator_pca = type(estimator)(**estimator.get_params())
    res_pca, best_est_pca, pred_pca, proba_pca = run_model(name, estimator_pca, grid, X_train_pca, y_train, X_test_pca, y_test)
    results_with_pca[name] = res_pca
    fitted_with_pca[name], preds_with_pca[name], probas_with_pca[name] = best_est_pca, pred_pca, proba_pca

## 8. Stacked Ensemble (SVM + Naive Bayes + Decision Tree -> Logistic Regression)

In [ ]:
def make_stack():
    base_learners = [
        ("svm", SVC(probability=True, random_state=rng)),
        ("nb", GaussianNB()),
        ("dt", DecisionTreeClassifier(random_state=rng)),
    ]
    return StackingClassifier(estimators=base_learners,
                               final_estimator=LogisticRegression(max_iter=5000, random_state=rng),
                               cv=skf)

for label, X_tr, X_te, results_dict, preds_dict, probas_dict in [
    ("No-PCA", X_train_std, X_test_std, results_no_pca, preds_no_pca, probas_no_pca),
    ("With-PCA", X_train_pca, X_test_pca, results_with_pca, preds_with_pca, probas_with_pca),
]:
    print(f"[{label}] Stacking")
    fold_scores = cross_val_score(make_stack(), X_tr, y_train, cv=skf, scoring="accuracy")
    metrics, pred, proba = test_metrics(make_stack(), X_tr, y_train, X_te, y_test)
    results_dict["Stacking"] = {"best_params": {"base_models": "SVM+NB+DT", "meta": "LogisticRegression"},
                                 "cv_fold_scores": fold_scores, "cv_avg": fold_scores.mean(),
                                 "cv_std": fold_scores.std(), "test_metrics": metrics}
    preds_dict["Stacking"], probas_dict["Stacking"] = pred, proba
    print(f"  CV_acc={fold_scores.mean():.4f}, test_acc={metrics['Accuracy']:.4f}")

## 9. Comparison Tables and Plots

In [ ]:
all_model_names = list(model_defs.keys()) + ["Stacking"]

comp_df = pd.DataFrame({
    name: {"No-PCA": results_no_pca[name]["cv_avg"], "With-PCA": results_with_pca[name]["cv_avg"]}
    for name in all_model_names
}).T
comp_df.plot(kind="bar", figsize=(11,6))
plt.title("5-Fold CV Accuracy: No-PCA vs. With-PCA (All Models)")
plt.ylabel("Avg CV Accuracy")
plt.xticks(rotation=45, ha="right")
plt.ylim(0.85, 1.0)
plt.legend(loc="lower right")
plt.show()
comp_df

In [ ]:
std_df = pd.DataFrame({
    name: {"No-PCA": results_no_pca[name]["cv_std"], "With-PCA": results_with_pca[name]["cv_std"]}
    for name in all_model_names
}).T
std_df.plot(kind="bar", figsize=(11,6))
plt.title("5-Fold CV Std Dev (Stability): No-PCA vs. With-PCA")
plt.ylabel("Std Dev of CV Accuracy across folds")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="upper right")
plt.show()
std_df

## 10. Confusion Matrices and ROC Curves (Selected Models)

In [ ]:
best_model_no_pca = max(all_model_names, key=lambda n: results_no_pca[n]["test_metrics"]["Accuracy"])
best_model_with_pca = max(all_model_names, key=lambda n: results_with_pca[n]["test_metrics"]["Accuracy"])
selected = sorted(set(["SVM", "Random Forest", "Logistic Regression", best_model_no_pca, best_model_with_pca]))
print("Selected models:", selected)

In [ ]:
fig, axes = plt.subplots(2, len(selected), figsize=(4*len(selected), 8))
for i, name in enumerate(selected):
    for row, (preds_dict, label) in enumerate([(preds_no_pca, "No-PCA"), (preds_with_pca, "With-PCA")]):
        ax = axes[row, i] if len(selected) > 1 else axes[row]
        cm = confusion_matrix(y_test, preds_dict[name])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=["mal","ben"], yticklabels=["mal","ben"])
        ax.set_title(f"{name} ({label})", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,6))
for name in selected:
    if probas_no_pca[name] is not None:
        fpr, tpr, _ = roc_curve(y_test, probas_no_pca[name])
        auc = roc_auc_score(y_test, probas_no_pca[name])
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves -- Selected Models (No-PCA)")
plt.legend(fontsize=8)
plt.show()

In [ ]:
plt.figure(figsize=(7,6))
for name in selected:
    if probas_with_pca[name] is not None:
        fpr, tpr, _ = roc_curve(y_test, probas_with_pca[name])
        auc = roc_auc_score(y_test, probas_with_pca[name])
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linestyle="--")
plt.plot([0,1],[0,1],"k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves -- Selected Models (With-PCA)")
plt.legend(fontsize=8)
plt.show()

## Summary

See the accompanying report (`Experiment6.pdf`) for the full hyperparameter grids, observation-question answers, and discussion of when PCA helps vs. hurts, grounded in these results.